# UIEC$^2$ End-to-End Notebook (Windows)

This single notebook contains:
- imports and setup
- dataloaders
- UIEC$^2$ model architecture
- training loop
- validation / test inference loop

It is adapted for Windows paths and runs on CUDA if available, otherwise CPU.

In [1]:
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.10.0+cu126
CUDA available: True


In [2]:
# Optional TensorBoard support
try:
    from torch.utils.tensorboard import SummaryWriter
except Exception:
    SummaryWriter = None

In [3]:
@dataclass
class Config:
    # Dataset roots (Windows-friendly)
    train_root: str = r".\DATA\Train"
    test_root: str = r".\DATA\Test"

    train_ann: str = r".\DATA\Train\train.txt"
    val_ann: str = r".\DATA\Test\test_time.txt"
    test_ann: str = r".\DATA\Test\test_time.txt"

    train_img_dir: str = r".\DATA\Train\train"
    train_gt_dir: str = r".\DATA\Train\gt"

    val_img_dir: str = r".\DATA\Test\test_time"
    val_gt_dir: str = r".\DATA\Test\gt"

    test_img_dir: str = r".\DATA\Test\test_time"
    test_gt_dir: str = r".\DATA\Test\gt"

    # Optimization
    batch_size: int = 2
    val_batch_size: int = 1
    num_workers: int = 0  # keep 0 on Windows unless you need multiprocessing
    lr: float = 1e-3
    betas: Tuple[float, float] = (0.9, 0.999)

    # LR schedule (from UIEC2Net config)
    total_epochs: int = 150
    lr_step_start: int = 100
    lr_step_end: int = 700
    linear_end_lr: float = 1e-5

    # Checkpoints / logs
    work_dir: str = r".\checkpoints\UIEC2Net\notebook"
    resume_from: str = ""   # set to checkpoint path to resume
    load_from: str = ""     # set to checkpoint path to load weights only

    # Testing output
    save_dir: str = r".\results\UIEC2Net_notebook"
    use_bytescale: bool = False

    # Reproducibility
    seed: int = 42

cfg = Config()
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.save_dir).mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# ---------------------------
# Utility: checkpoints / image save
# ---------------------------
def remove_prefix(state_dict, prefix="module."):
    out = {}
    for k, v in state_dict.items():
        out[k[len(prefix):] if k.startswith(prefix) else k] = v
    return out

def save_checkpoint(path, model, optimizer=None, epoch=0, iters=0):
    state_dict = {k: v.cpu() for k, v in model.state_dict().items()}
    ckpt = {
        "meta": {"epoch": epoch, "iter": iters},
        "state_dict": state_dict,
    }
    if optimizer is not None:
        ckpt["optimizer"] = optimizer.state_dict()
    torch.save(ckpt, path)

def load_checkpoint(path, model, optimizer=None, resume_optimizer=False, map_location=None):
    ckpt = torch.load(path, map_location=map_location or device)
    sd = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    sd = remove_prefix(sd, "module.")
    model.load_state_dict(sd, strict=True)

    start_epoch = 1
    start_iter = 0
    if "meta" in ckpt:
        start_epoch = ckpt["meta"].get("epoch", 1)
        start_iter = ckpt["meta"].get("iter", 0)
    if optimizer is not None and resume_optimizer and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return start_epoch, start_iter

def tensor_to_uint8_image(x: torch.Tensor):
    # x: [B, C, H, W] in [0,1], use first sample
    arr = x[0].detach().cpu().float().numpy().transpose(1, 2, 0)
    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
    return arr

def save_image_np(np_img: np.ndarray, out_path: str):
    Image.fromarray(np_img).save(out_path)

In [5]:
# ---------------------------
# Loss: SSIMLoss (same idea as project)
# ---------------------------
from math import exp

def gaussian(window_size, sigma):
    gauss = torch.tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()

def create_window(window_size, channel):
    _1d = gaussian(window_size, 1.5).unsqueeze(1)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2d.expand(channel, 1, window_size, window_size).contiguous()
    return window

def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))

    if size_average:
        return ssim_map.mean()
    return ssim_map.mean(1).mean(1).mean(1)

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, size_average=True, loss_weight=1.0):
        super().__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = create_window(window_size, self.channel)
        self.loss_weight = loss_weight

    def forward(self, img1, img2):
        _, channel, _, _ = img1.size()
        if channel != self.channel or self.window.dtype != img1.dtype:
            self.window = create_window(self.window_size, channel).to(img1.device).type_as(img1)
            self.channel = channel
        window = self.window.to(img1.device).type_as(img1)
        return self.loss_weight * (1.0 - _ssim(img1, img2, window, self.window_size, channel, self.size_average))

In [6]:
# ---------------------------
# Dataset / transforms
# ---------------------------
class Compose:
    def __init__(self, transforms_list):
        self.transforms_list = transforms_list

    def __call__(self, sample):
        for t in self.transforms_list:
            sample = t(sample)
        return sample

class LoadImageFromFile:
    def __init__(self, get_gt=True):
        self.get_gt = get_gt

    def __call__(self, sample):
        image = Image.open(sample["image_path"]).convert("RGB")
        out = {"image": image, "image_path": sample["image_path"], "image_id": Path(sample["image_path"]).stem}
        if self.get_gt:
            gt = Image.open(sample["gt_path"]).convert("RGB")
            out["gt"] = gt
            out["gt_path"] = sample["gt_path"]
        return out

# ✅ NEW: Resize both image and gt to a fixed size
class ResizeToFixed:
    def __init__(self, size=(480, 640)):  # (height, width)
        self.size = size  # PIL expects (width, height) so we flip below

    def __call__(self, sample):
        w, h = self.size[1], self.size[0]  # PIL uses (width, height)
        sample["image"] = sample["image"].resize((w, h), Image.BILINEAR)
        if "gt" in sample:
            sample["gt"] = sample["gt"].resize((w, h), Image.BILINEAR)
        return sample

class RandomFlip:
    def __init__(self, flip_ratio=0.5):
        self.flip_ratio = flip_ratio

    def __call__(self, sample):
        if random.random() < self.flip_ratio:
            sample["image"] = sample["image"].transpose(Image.FLIP_LEFT_RIGHT)
            if "gt" in sample:
                sample["gt"] = sample["gt"].transpose(Image.FLIP_LEFT_RIGHT)
        return sample

class ImageToTensor:
    def __init__(self):
        self.to_tensor = transforms.ToTensor()

    def __call__(self, sample):
        sample["image"] = self.to_tensor(sample["image"])
        if "gt" in sample:
            sample["gt"] = self.to_tensor(sample["gt"])
        return sample

class AlignedDataset(Dataset):
    def __init__(self, ann_file: str, img_dir: str, gt_dir: str, pipeline=None, test_mode=False):
        self.ann_file = ann_file
        self.img_dir = img_dir
        self.gt_dir = gt_dir
        self.pipeline = pipeline
        self.test_mode = test_mode
        self.data_infos = self._load_annotations()

    def _load_annotations(self):
        infos = []
        with open(self.ann_file, "r", encoding="utf-8", errors="ignore") as f:
            names = [x.strip() for x in f.readlines() if x.strip()]
        for name in names:
            infos.append({
                "image_path": str(Path(self.img_dir) / name),
                "gt_path": str(Path(self.gt_dir) / name),
            })
        return infos

    def __len__(self):
        return len(self.data_infos)

    def __getitem__(self, idx):
        sample = dict(self.data_infos[idx])
        if self.pipeline is not None:
            sample = self.pipeline(sample)
        return sample

In [7]:
def build_dataloaders(cfg: Config):
    train_pipeline = Compose([
        LoadImageFromFile(get_gt=True),
        ResizeToFixed(size=(480, 640)),   # ✅ ADD THIS
        RandomFlip(flip_ratio=0.5),
        ImageToTensor(),
    ])
    val_pipeline = Compose([
        LoadImageFromFile(get_gt=True),
        ResizeToFixed(size=(480, 640)),   # ✅ ADD THIS
        ImageToTensor(),
    ])
    test_pipeline = Compose([
        LoadImageFromFile(get_gt=False),
        ResizeToFixed(size=(480, 640)),   # ✅ ADD THIS
        ImageToTensor(),
    ])
    # ... rest stays the same

    train_ds = AlignedDataset(
        ann_file=cfg.train_ann,
        img_dir=cfg.train_img_dir,
        gt_dir=cfg.train_gt_dir,
        pipeline=train_pipeline,
        test_mode=False,
    )

    val_ds = AlignedDataset(
        ann_file=cfg.val_ann,
        img_dir=cfg.val_img_dir,
        gt_dir=cfg.val_gt_dir,
        pipeline=val_pipeline,
        test_mode=True,
    )

    test_ds = AlignedDataset(
        ann_file=cfg.test_ann,
        img_dir=cfg.test_img_dir,
        gt_dir=cfg.test_gt_dir,
        pipeline=test_pipeline,
        test_mode=True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.val_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.val_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_dataloaders(cfg)
print("Train batches:", len(train_loader), "Val batches:", len(val_loader), "Test batches:", len(test_loader))

Train batches: 356 Val batches: 178 Test batches: 178


In [8]:
# ---------------------------
# Model blocks: RGB <-> HSV
# ---------------------------
class RGB2HSV(nn.Module):
    def forward(self, rgb):
        b, c, h, w = rgb.size()
        r, g, bch = rgb[:, 0], rgb[:, 1], rgb[:, 2]
        v, max_index = torch.max(rgb, dim=1)
        min_rgb = torch.min(rgb, dim=1)[0]
        delta = v - min_rgb

        s = delta / (v + 1e-4)
        hval = torch.zeros_like(rgb[:, 0])

        mark = max_index == 0
        hval[mark] = 60.0 * (g[mark] - bch[mark]) / (delta[mark] + 1e-4)
        mark = max_index == 1
        hval[mark] = 120.0 + 60.0 * (bch[mark] - r[mark]) / (delta[mark] + 1e-4)
        mark = max_index == 2
        hval[mark] = 240.0 + 60.0 * (r[mark] - g[mark]) / (delta[mark] + 1e-4)

        hval[hval < 0] += 360.0
        hval = (hval % 360.0) / 360.0
        hsv = torch.cat([hval.view(b, 1, h, w), s.view(b, 1, h, w), v.view(b, 1, h, w)], dim=1)
        return hsv

def function_delta(x):
    return torch.clamp(x, min=0, max=60)

class HSV2RGB(nn.Module):
    def forward(self, hsv):
        b, c, h, w = hsv.size()
        hh, s, v = hsv[:, 0], hsv[:, 1], hsv[:, 2]
        hh = ((hh * 360.0) % 360.0) / 360.0

        vs = (v * s) / 60.0
        r1 = function_delta(hh * 360.0 - 60)
        r2 = function_delta(hh * 360.0 - 240)
        g1 = function_delta(hh * 360.0)
        g2 = function_delta(hh * 360.0 - 180)
        b1 = function_delta(hh * 360.0 - 120)
        b2 = function_delta(hh * 360.0 - 300)

        one_minus_s = 1.0 - s

        r = (v + vs * r1) - vs * r2
        g = (v * one_minus_s + vs * g1) - vs * g2
        bch = (v * one_minus_s + vs * b1) - vs * b2

        return torch.cat([
            r.view(b, 1, h, w),
            g.view(b, 1, h, w),
            bch.view(b, 1, h, w),
        ], dim=1)

In [9]:
# ---------------------------
# UIEC2Net architecture
# ---------------------------
def sgn_m(x):
    return torch.clamp(x, 0.0, 1.0)

def piece_function_org(x_m, para_m, m):
    b, c, h, w = x_m.shape
    r_m = para_m[:, 0].view(b, c, 1, 1).expand(b, c, h, w)
    for i in range(m - 1):
        delta = (para_m[:, i + 1] - para_m[:, i]).view(b, c, 1, 1).expand(b, c, h, w)
        r_m = r_m + delta * sgn_m(m * x_m - i * torch.ones_like(x_m))
    return r_m

class UIEC2Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.rgb2hsv = RGB2HSV()
        self.hsv2rgb = HSV2RGB()

        # RGB branch
        self.rgb_con1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.rgb_con2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.rgb_con3 = nn.Conv2d(64, 64, 3, 1, 1)
        self.rgb_con4 = nn.Conv2d(64, 64, 3, 1, 1)
        self.rgb_con5 = nn.Conv2d(64, 64, 3, 1, 1)
        self.rgb_con6 = nn.Conv2d(64, 64, 3, 1, 1)
        self.rgb_con7 = nn.Conv2d(64, 64, 1, 1, 0)
        self.rgb_in1 = nn.InstanceNorm2d(64)
        self.rgb_in2 = nn.InstanceNorm2d(64)
        self.rgb_in3 = nn.InstanceNorm2d(64)
        self.rgb_in4 = nn.InstanceNorm2d(64)
        self.rgb_in5 = nn.InstanceNorm2d(64)
        self.rgb_in6 = nn.InstanceNorm2d(64)
        self.rgb_down = nn.LeakyReLU(inplace=True)
        self.rgb_up = nn.ReLU(inplace=True)

        # HSV branch
        self.M = 11
        nf = 64
        self.e_conv1 = nn.Conv2d(6, nf, 3, 1, 1, bias=True)
        self.e_conv2 = nn.Conv2d(nf, nf, 3, 1, 1, bias=True)
        self.e_conv3 = nn.Conv2d(nf, nf, 3, 1, 1, bias=True)
        self.e_conv4 = nn.Conv2d(nf, nf, 3, 1, 1, bias=True)
        self.e_conv7 = nn.Conv2d(nf, nf, 3, 1, 1, bias=True)
        self.e_fc = nn.Linear(nf, 44)
        self.relu = nn.LeakyReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        # Confidence branch
        self.con1 = nn.Conv2d(9, 64, 3, 1, 1)
        self.con2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.con3 = nn.Conv2d(64, 64, 3, 1, 1)
        self.con4 = nn.Conv2d(64, 64, 3, 1, 1)
        self.con5 = nn.Conv2d(64, 64, 3, 1, 1)
        self.con6 = nn.Conv2d(64, 64, 3, 1, 1)
        self.con7 = nn.Conv2d(64, 6, 1, 1, 0)
        self.in1 = nn.InstanceNorm2d(64)
        self.in2 = nn.InstanceNorm2d(64)
        self.in3 = nn.InstanceNorm2d(64)
        self.in4 = nn.InstanceNorm2d(64)
        self.in5 = nn.InstanceNorm2d(64)
        self.in6 = nn.InstanceNorm2d(64)
        self.conf_relu = nn.LeakyReLU(inplace=True)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        h = self.rgb_down(self.rgb_in1(self.rgb_con1(x)))
        h = self.rgb_down(self.rgb_in2(self.rgb_con2(h)))
        h = self.rgb_down(self.rgb_in3(self.rgb_con3(h)))
        h = self.rgb_down(self.rgb_in4(self.rgb_con4(h)))
        h = self.rgb_up(self.rgb_in5(self.rgb_con5(h)))
        h = self.rgb_up(self.rgb_in6(self.rgb_con6(h)))
        rgb_out = torch.sigmoid(self.rgb_con7(h))[:, 0:3]

        hsv_from_rgb = self.rgb2hsv(rgb_out)
        hsv_input = torch.cat([hsv_from_rgb, hsv_from_rgb], dim=1)

        bsz = hsv_input.size(0)
        x1 = self.relu(self.e_conv1(hsv_input)); x1 = self.maxpool(x1)
        x2 = self.relu(self.e_conv2(x1)); x2 = self.maxpool(x2)
        x3 = self.relu(self.e_conv3(x2)); x3 = self.maxpool(x3)
        x4 = self.relu(self.e_conv4(x3))
        xr = self.relu(self.e_conv7(x4))
        xr = self.avgpool(xr).view(bsz, -1)
        xr = self.e_fc(xr)

        H, S, V, H2S = torch.split(xr, self.M, dim=1)
        H_in = hsv_input[:, 0:1]
        S_in = hsv_input[:, 1:2]
        V_in = hsv_input[:, 2:3]

        H_out = piece_function_org(H_in, H, self.M)
        S_out1 = piece_function_org(S_in, S, self.M)
        V_out = piece_function_org(V_in, V, self.M)
        S_out2 = piece_function_org(H_in, H2S, self.M)

        S_out = torch.clamp((S_out1 + S_out2) / 2.0, 0.0, 1.0)
        V_out = torch.clamp(V_out, 0.0, 1.0)

        hsv_out = torch.cat([H_out, S_out, V_out], dim=1)
        hsv_out_rgb = self.hsv2rgb(hsv_out)

        conf_in = torch.cat([x, rgb_out, hsv_out_rgb], dim=1)
        h = self.conf_relu(self.in1(self.con1(conf_in)))
        h = self.conf_relu(self.in2(self.con2(h)))
        h = self.conf_relu(self.in3(self.con3(h)))
        h = self.conf_relu(self.in4(self.con4(h)))
        h = self.conf_relu(self.in5(self.con5(h)))
        h = self.conf_relu(self.in6(self.con6(h)))

        conf = torch.sigmoid(self.con7(h))
        conf_rgb = conf[:, 0:3]
        conf_hsv = conf[:, 3:6]

        out = 0.5 * conf_rgb * rgb_out + 0.5 * conf_hsv * hsv_out_rgb
        return out

model = UIEC2Net().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 534,962


In [10]:
# ---------------------------
# Optimizer + LR scheduler
# ---------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=cfg.betas)
criterion_ssim = SSIMLoss(window_size=11, size_average=True, loss_weight=1.0).to(device)

# Reproduce UIEC2Net "Epoch linear" schedule behavior
base_lr = cfg.lr

def lambda_rule(epoch_idx):
    epoch_num = epoch_idx + 1
    if cfg.lr_step_start <= epoch_num <= cfg.lr_step_end:
        return 1.0 - max(0, epoch_num - cfg.lr_step_start) / float(cfg.lr_step_end - cfg.lr_step_start + 1)
    if epoch_num >= cfg.lr_step_end:
        return cfg.linear_end_lr / base_lr
    return 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_rule)

start_epoch = 1
global_iter = 0
if cfg.resume_from:
    start_epoch, global_iter = load_checkpoint(cfg.resume_from, model, optimizer=optimizer, resume_optimizer=True)
    print("Resumed from:", cfg.resume_from, "start_epoch:", start_epoch, "global_iter:", global_iter)
elif cfg.load_from:
    load_checkpoint(cfg.load_from, model, optimizer=None, resume_optimizer=False)
    print("Loaded weights from:", cfg.load_from)

In [11]:
# ---------------------------
# Train / Validate / Test
# ---------------------------

def train_one_epoch(model, loader, optimizer, criterion, epoch, writer=None):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc=f"Train Epoch {epoch}", leave=False)

    global global_iter
    for batch in pbar:
        inputs = batch["image"].to(device, non_blocking=True)
        targets = batch["gt"].to(device, non_blocking=True)

        preds = model(inputs)
        loss_ssim = criterion(preds, targets)
        loss = loss_ssim

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        global_iter += 1
        running_loss += loss.item()

        if writer is not None:
            writer.add_scalar("train/loss_ssim", loss_ssim.item(), global_iter)
            writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], global_iter)

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.6f}")

    return running_loss / max(len(loader), 1)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    val_loss = 0.0
    for batch in tqdm(loader, desc="Validate", leave=False):
        inputs = batch["image"].to(device, non_blocking=True)
        # val pipeline has no gt in original config; if gt exists, compute loss
        if "gt" in batch:
            targets = batch["gt"].to(device, non_blocking=True)
            preds = model(inputs)
            loss = criterion(preds, targets)
            val_loss += loss.item()
        else:
            _ = model(inputs)
    return val_loss / max(len(loader), 1)

@torch.no_grad()
def run_test_and_save(model, loader, out_dir):
    model.eval()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for batch in tqdm(loader, desc="Test/Save", leave=False):
        inputs = batch["image"].to(device, non_blocking=True)
        preds = model(inputs)

        names = batch["image_id"]
        for bi in range(preds.size(0)):
            img = preds[bi:bi+1]
            np_img = np.clip(img[0].detach().cpu().numpy().transpose(1, 2, 0) * 255.0, 0, 255).astype(np.uint8)
            save_image_np(np_img, str(out_dir / f"{names[bi]}.png"))

In [12]:
# ---------------------------
# Main training run
# ---------------------------
def run_training(cfg: Config, model, optimizer, scheduler, criterion, train_loader, val_loader):
    writer = SummaryWriter(log_dir=str(Path(cfg.work_dir) / "tb_logs")) if SummaryWriter is not None else None

    best_val = float("inf")
    for epoch in range(start_epoch, cfg.total_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, epoch, writer=writer)

        # Original project val pipeline can be gt-free; this still runs safely.
        val_loss = validate(model, val_loader, criterion)

        scheduler.step()

        print(f"Epoch {epoch:04d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | lr={optimizer.param_groups[0]['lr']:.7f}")

        # Save latest
        save_checkpoint(str(Path(cfg.work_dir) / "latest.pth"), model, optimizer, epoch=epoch, iters=global_iter)

        # Save periodic
        if (epoch % 20 == 0) or (epoch == cfg.total_epochs):
            save_checkpoint(str(Path(cfg.work_dir) / f"epoch_{epoch}.pth"), model, optimizer, epoch=epoch, iters=global_iter)

        # Best by val loss (if val_loss is 0 due to no-gt val, this will just keep first epoch)
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint(str(Path(cfg.work_dir) / "best.pth"), model, optimizer, epoch=epoch, iters=global_iter)

    if writer is not None:
        writer.close()

In [13]:
# Uncomment to train (can take a long time with total_epochs=1000)
run_training(cfg, model, optimizer, scheduler, criterion_ssim, train_loader, val_loader)

Epoch 0001 | train_loss=0.165879 | val_loss=0.125439 | lr=0.0010000


Epoch 0002 | train_loss=0.113608 | val_loss=0.116234 | lr=0.0010000


Epoch 0003 | train_loss=0.106022 | val_loss=0.108857 | lr=0.0010000


Epoch 0004 | train_loss=0.103786 | val_loss=0.111091 | lr=0.0010000


Epoch 0005 | train_loss=0.099985 | val_loss=0.106485 | lr=0.0010000


...

Epoch 0146 | train_loss=0.064246 | val_loss=0.086498 | lr=0.0009218


Epoch 0147 | train_loss=0.063870 | val_loss=0.086878 | lr=0.0009201


Epoch 0148 | train_loss=0.062996 | val_loss=0.086487 | lr=0.0009185


Epoch 0149 | train_loss=0.062805 | val_loss=0.086245 | lr=0.0009168


Epoch 0150 | train_loss=0.062824 | val_loss=0.086044 | lr=0.0009151


In [17]:
# ---------------------------
# Inference / testing
# ---------------------------
# If you want to test from a saved checkpoint, set ckpt_path accordingly.
ckpt_path = str(Path(cfg.work_dir) / "latest.pth")
if Path(ckpt_path).exists():
    _ = load_checkpoint(ckpt_path, model, optimizer=None, resume_optimizer=False)
    print("Loaded for test:", ckpt_path)
else:
    print("Checkpoint not found, using current model weights.")

run_test_and_save(model, test_loader, cfg.save_dir)
print("Saved test outputs to:", cfg.save_dir)

Loaded for test: checkpoints\UIEC2Net\notebook\latest.pth


Saved test outputs to: .\results\UIEC2Net_notebook


In [ ]:
# ---------------------------
# Evaluation: PSNR + Visual Comparison
# ---------------------------
import math
import matplotlib.pyplot as plt
import torch
import numpy as np
from pathlib import Path

# ── PSNR helper ──────────────────────────────────────────────────────────────
def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    """pred, target: [C, H, W] float tensors in [0, 1]"""
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float("inf")
    return 10 * math.log10(1.0 / mse)


# ── Tensor → numpy for display ───────────────────────────────────────────────
def to_np(t: torch.Tensor) -> np.ndarray:
    """[C, H, W] float tensor → HWC uint8"""
    return (t.permute(1, 2, 0).cpu().float().clamp(0, 1).numpy() * 255).astype(np.uint8)


# ── Full evaluation loop ─────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_model(model, loader, num_examples=5):
    model.eval()

    psnr_list = []
    examples_shown = 0

    # How many examples to show (spread across dataset)
    total = len(loader.dataset)
    show_every = max(1, total // num_examples)

    fig_rows = []  # collect (input_np, pred_np, gt_np, psnr_val)

    for idx, batch in enumerate(tqdm(loader, desc="Evaluating")):
        inputs  = batch["image"].to(device)   # [B, 3, H, W]
        targets = batch["gt"].to(device)       # [B, 3, H, W]

        preds = model(inputs)                  # [B, 3, H, W]

        for b in range(inputs.size(0)):
            global_idx = idx * loader.batch_size + b
            psnr_val = compute_psnr(preds[b], targets[b])
            psnr_list.append(psnr_val)

            # Collect examples evenly spaced through dataset
            if examples_shown < num_examples and global_idx % show_every == 0:
                fig_rows.append((
                    to_np(inputs[b]),
                    to_np(preds[b]),
                    to_np(targets[b]),
                    psnr_val,
                    batch["image_id"][b] if isinstance(batch["image_id"], list) else batch["image_id"][b]
                ))
                examples_shown += 1

    # ── Print summary ─────────────────────────────────────────────────────────
    avg_psnr = sum(psnr_list) / len(psnr_list)
    min_psnr = min(psnr_list)
    max_psnr = max(psnr_list)

    print("=" * 50)
    print(f"  Results over {len(psnr_list)} images")
    print(f"  Avg PSNR : {avg_psnr:.2f} dB")
    print(f"  Min PSNR : {min_psnr:.2f} dB")
    print(f"  Max PSNR : {max_psnr:.2f} dB")
    print("=" * 50)

    # ── Visual comparison grid ────────────────────────────────────────────────
    n = len(fig_rows)
    fig, axes = plt.subplots(n, 3, figsize=(14, 4 * n))

    # Handle case where n=1 (axes won't be 2D)
    if n == 1:
        axes = [axes]

    col_titles = ["Input (degraded)", "Model Output (enhanced)", "Ground Truth"]
    for col, title in enumerate(col_titles):
        axes[0][col].set_title(title, fontsize=13, fontweight="bold")

    for row, (inp, pred, gt, psnr_val, img_id) in enumerate(fig_rows):
        for col, img in enumerate([inp, pred, gt]):
            axes[row][col].imshow(img)
            axes[row][col].axis("off")
        # PSNR label on the enhanced image column
        axes[row][1].set_xlabel(f"PSNR: {psnr_val:.2f} dB", fontsize=11, color="green")
        axes[row][0].set_ylabel(f"{img_id}", fontsize=9, rotation=0, labelpad=60, va="center")

    plt.suptitle("UIEC²Net — Visual Evaluation", fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    return avg_psnr, psnr_list


# ── Run it ────────────────────────────────────────────────────────────────────
# Load best checkpoint if available
ckpt_path = str(Path(cfg.work_dir) / "best.pth")
if Path(ckpt_path).exists():
    load_checkpoint(ckpt_path, model)
    print("Loaded:", ckpt_path)
else:
    print("No checkpoint found — using current model weights.")

avg_psnr, all_psnrs = evaluate_model(model, val_loader, num_examples=20)

Loaded: checkpoints\UIEC2Net\notebook\best.pth


Evaluating: 100%|██████████| 178/178 [00:17<00:00, 10.32it/s]


  Results over 178 images
  Avg PSNR : 23.36 dB
  Min PSNR : 12.73 dB
  Max PSNR : 33.12 dB


In [37]:
import torch
import os

# =========================
# 1) SET SAVE PATHS
# =========================
save_dir = "./saved_model"
os.makedirs(save_dir, exist_ok=True)

torchscript_path = os.path.join(save_dir, "uiec2net_torchscript.pt")
state_dict_path  = os.path.join(save_dir, "uiec2net_state_dict.pth")

# =========================
# 2) PUT MODEL IN EVAL MODE
# =========================
model.eval()

# =========================
# 3) SAVE STATE_DICT (backup)
# =========================
torch.save({
    "model_state_dict": model.state_dict(),
}, state_dict_path)

print(f"✅ Saved state_dict to: {state_dict_path}")

# =========================
# 4) SAVE TORCHSCRIPT (portable)
# =========================
# Create a dummy input with the same shape you trained on
# (Batch=1, Channels=3, Height, Width)
dummy_input = torch.randn(1, 3, 256, 256).to(next(model.parameters()).device)

# Convert model to TorchScript
with torch.no_grad():
    scripted_model = torch.jit.trace(model, dummy_input)

# Save TorchScript model
scripted_model.save(torchscript_path)

print(f"🔥 Saved TorchScript (portable) to: {torchscript_path}")
print("\nNow you can load this .pt file on ANY PC without needing the model class code.")


✅ Saved state_dict to: ./saved_model\uiec2net_state_dict.pth
🔥 Saved TorchScript (portable) to: ./saved_model\uiec2net_torchscript.pt

Now you can load this .pt file on ANY PC without needing the model class code.
